# Page 8 — "Best-of-N and Verifiers": does judging beat counting, and what does it cost to find out

Page 7 kept the answer that came up most. This page keeps the answer a **verifier** likes best, and
then asks the only question that matters: does that beat counting?

It is not a rhetorical question. Best-of-N accuracy is `r x coverage(N)`, where `r` is how often the
verifier picks a correct candidate when one is in the pile. Majority voting's accuracy is a measured
number. So the reliability a verifier must reach to draw with counting is
`majority(N) / coverage(N)`, and that ratio is **the bar**: a quantity this notebook measures before
it scores anything, on the same candidates, so that every later claim is checked against a threshold
that was fixed in advance.

Three rules carry the page, and the first figure is all three on one axis:

| rule | what it knows | cost |
|---|---|---|
| one chain (plain CoT) | nothing; it is the single-sample baseline | free |
| majority vote | which answer recurs, which is page 7's rule unchanged | free |
| **best-of-N** | a verifier's judgement of each candidate | one forward pass per candidate |
| coverage (oracle) | the ground truth, so it is the ceiling all three share | free |

Two scorers from ch05, `heuristic_score` and `avg_logprob_answer`, are kept as **diagnostics**
rather than as the payoff. Neither reads whether the reasoning is right, and the table in section 11
is where that shows.

The plan, the measured bar and the go/no-go gate live in
[`docs/page_8_measured_payoff.md`](../docs/page_8_measured_payoff.md).
Writes `out/page_08/`: `traces.jsonl` (only in `GENERATE`), `scores.jsonl`, `derived.json`,
`manifest.json`.


## 0. How to run this

The modes are the phases of the experiment, not tiers of the same run.

| `RUN_MODE` | What it does | Hardware | Time |
|---|---|---|---|
| **`REPLAY`** *(default)* | every figure from the candidates and scores already on disk | **none** | ~1 min |
| `SCORE` | **phase 0.** Runs the verifier tiers over a candidate pool that already exists. No generation. This is the decision point | ~8 GB GPU | tens of minutes |
| `GENERATE` | **phase 1.** Draws page 8's own pool, then scores it | 16 GB GPU | hours |

**Run `SCORE` first and read section 12.** Phase 1 is only worth its hours if a verifier clears the
bar in phase 0, and phase 0 is free of generation because page 7 already paid for a pool.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

`REPLAY` and `SCORE` need a pool. Either run page 7 first, or point `PAGE_07_TRACES_URL` at its
published `traces.jsonl`. `GENERATE` needs neither: it makes its own.


# Part 1 — The candidates, and the bar a verifier has to clear

Loading the pool, restating page 7's vote on it, and measuring what any verifier must beat. Nothing
here loads a model and nothing here draws.


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU) | "SCORE" (phase 0) | "GENERATE" (phase 1)
SEED = 123

# Where the candidates come from. "page_07" is the phase 0 pilot bed: a pool that has
# already been generated and paid for. "own" is page 8's own run, which GENERATE writes.
POOL_SOURCE = "page_07"   # "page_07" | "own"

# An item has to carry at least this many candidates at a configuration for that
# configuration to be plotted, or the N axis means a different thing on different items.
MIN_POOL = 8

# One chain length, across the whole page. Page 7's file also holds configurations at
# shorter caps, because splitting a fixed budget N ways is page 7's axis. Mixing them in
# here would confound "more candidates" and "hotter candidates" with "shorter candidates",
# which is the one thing this page must not do. None picks the deepest cap on disk.
CHAIN_CAP = None

# The N axis is built by drawing subsets from a fixed pool, so chain LENGTH is held fixed
# and the only thing changing is how many candidates the rule may look at. Anything else
# would confound "more candidates" with "shorter candidates", which is page 7's axis.
N_RESAMPLES = 64          # random subsets per (item, N); the sampling error on each point
N_BOOTSTRAP = 1000        # resamples of the ITEM set, which is what the intervals are over

# The buckets are defined by what the vote does and then measured on the same items, so
# the pile is halved: odd-indexed chains name the bucket, even-indexed ones are what the
# curves are computed from. False is the circular version, useful only to show what the
# circularity was worth.
BUCKET_HOLDOUT = True

ITEM_LIMIT = None         # e.g. 25 for a fast pass over every cell; None uses every item

# ---------------------------------------------------------------------------
# The verifiers, in the order they are tried. The first one to clear the bar is the
# page's line and the rest are diagnostics. Tiers exist because "a good enough verifier"
# is the page's premise and the cheapest sufficient one is the honest choice.
#
#   kind="yesno"  a chat model asked whether it believes the solution, read as
#                 log P(Yes) - log P(No) at one position. No generation, no verdict to
#                 parse, and a continuous score that still ranks when the model hedges.
#   kind="rm"     a trained reward model, scored by its own head. This is Cobbe et al.'s
#                 actual construction rather than a chat model imitating it.
#
# `self` is the control and is expected to fail: a model grading its own work shares
# every error mode that produced it. Its failure is a sentence on the page, not a gap.
# ---------------------------------------------------------------------------
# `solves` is that checkpoint's OWN accuracy on these same items at this same cap,
# measured by page 6. It is not decoration: a judge that solves more of the item set than
# the pile's writer does is a better GENERATOR, and beating the vote with one is a much
# weaker claim than the page's title implies. Section 4 prints the comparison.
VERIFIERS = [
    # The writer grading its own work. Same size by construction, free because the
    # checkpoint is already resident, and the control the other tiers are read against.
    dict(name="self", kind="yesno", model=None, quant=None, solves=0.43, enabled=True),
    # SMALLER than the writer, and much worse at the task: page 6 has it at 22% against
    # the 3B's 43%. This is the page's thesis in its strongest form. A model that can
    # barely solve these problems, selecting better than the crowd counts, is
    # "verification is easier than generation" demonstrated rather than quoted.
    dict(name="qwen1.5b", kind="yesno", model="Qwen/Qwen2.5-1.5B-Instruct", quant=None,
         solves=0.22, enabled=True),
    # Stronger than the writer, and the rung to read with care: it solves 60% of these
    # items alone, which is more than the 3B's vote manages at N=16. It answers "how far
    # can a better judge go" and it cannot be the page's headline without the page saying
    # that out loud.
    dict(name="qwen7b", kind="yesno", model="Qwen/Qwen2.5-7B-Instruct", quant="8bit",
         solves=0.60, enabled=True),
    # A trained reward model: the conceptually right answer, because it is trained to
    # verify rather than to generate, so it is not a better generator wearing a judge's
    # hat. Set enabled=True and name a checkpoint verified against the hub. An OUTCOME
    # reward model keeps page 9's process-supervision story intact; a process reward model
    # would win more easily and spend page 9's subject to do it.
    dict(name="orm", kind="rm", model=None, quant="8bit", solves=None, enabled=False),
]

# Not a math-tuned checkpoint, and the reason is in this repo's own ablations rather than
# in taste: experiment 2 found every Qwen2.5-Math model at 0% compliance with a plain
# instruction, and a one-word Yes/No verdict is exactly the instruction-following they are
# worst at. Their competence on these items has also never been measured here, while all
# three general checkpoints above carry a page 6 number.

# GENERATE only. Page 8's own pool: its own run_id, the deck's instrument, and a
# temperature GRID rather than a point, because the grid is the finding.
GEN = dict(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    dataset="math500_l45",
    n_items=100,
    chain_cap=2048,
    top_p=0.95,
    # (temperature, chains per item). Page 7's pilot pool is 100 items at T=0.3 and
    # 30 items at each hotter temperature, so the hot end of figure 2 is a shape rather
    # than a measurement. This grid's first job is to fix that, and its second is to give
    # best-of-N the wide pool the bar falls with. Deep where the bar is lowest, shallower
    # where it is not: the full grid at the full depth is 25,600 generations.
    grid=[(0.3, 16), (0.6, 16), (0.9, 32), (1.2, 64)],
    group=8,              # chains per generate call
)

# Where REPLAY reads what it replays.
PAGE_07_TRACES_URL = None
TRACES_URL = None
SCORES_URL = None

assert RUN_MODE in ("REPLAY", "SCORE", "GENERATE"), RUN_MODE
print(f"mode={RUN_MODE}  pool={POOL_SOURCE}  "
      f"verifiers={[v['name'] for v in VERIFIERS if v['enabled']]}")


In [ ]:
import inspect
import json
import math
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    render_prompt, extract_final_candidate, grade_answer)
from reasoning_from_scratch.ch05 import heuristic_score, avg_logprob_answer

OUT_DIR = Path("out/page_08")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_07_DIR = Path("out/page_07")

P7_TRACES = PAGE_07_DIR / "traces.jsonl"
P7_DERIVED = PAGE_07_DIR / "derived.json"
OWN_TRACES = OUT_DIR / "traces.jsonl"
SCORES_PATH = OUT_DIR / "scores.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])


## 1. The pool, deduplicated on the way in

One model, one item set, one chain length, and **one configuration per temperature**. The
temperature is an axis on this page rather than a fixed choice, because it is the knob that decides
how much there is to count and how much there is to select from.

**Deduplication is not defensive tidying.** `TraceStore` has no write lock, so two processes on one
`run_id` silently produce duplicate records. Page 7's file has held 480 of them. A duplicated
candidate is counted twice by the vote and drawn twice by the subsampler, which biases every number
below, so the load asserts uniqueness rather than trusting it.


In [ ]:
def load_pool(path, url=None, label=""):
    """Every record in one trace file, keyed and deduplicated. Returns (records, run_id)."""
    if url and not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"fetching {label} from {url}")
        path.write_text(requests.get(url, timeout=600).text, encoding="utf-8")
    assert path.exists(), (
        f"no candidates at {path}. Run the notebook that makes them, or set the matching "
        f"URL in the knobs cell.")

    rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
    assert rows, f"{path} is empty"
    run_id = Counter(r["run_id"] for r in rows).most_common(1)[0][0]
    rows = [r for r in rows if r["run_id"] == run_id]

    seen, dropped = {}, 0
    for r in rows:
        k = (r["model_id"], r["config_id"], r["problem_id"], r["sample_index"])
        if k in seen:
            dropped += 1
            assert seen[k]["trace"] == r["trace"], (
                f"two DIFFERENT generations share the key {k}. That is not a duplicated "
                f"write, it is two runs sharing a run_id, and neither can be trusted.")
            continue
        seen[k] = r
    if dropped:
        print(f"  {dropped} duplicate records dropped (identical traces under identical "
              f"keys: the unlocked-TraceStore trap)")
    return list(seen.values()), run_id


if POOL_SOURCE == "page_07":
    raw, SOURCE_RUN_ID = load_pool(P7_TRACES, PAGE_07_TRACES_URL, "page 7's candidates")
    POOL_ORIGIN = "page 7's generations, re-used as the phase 0 pilot bed"
else:
    raw, SOURCE_RUN_ID = load_pool(OWN_TRACES, TRACES_URL, "page 8's own candidates")
    POOL_ORIGIN = "page 8's own generations"

MODEL_ID = Counter(r["model_id"] for r in raw).most_common(1)[0][0]
raw = [r for r in raw if r["model_id"] == MODEL_ID]
print(f"source run  {SOURCE_RUN_ID}")
print(f"model       {MODEL_ID}")
print(f"records     {len(raw):,}  ({POOL_ORIGIN})")


In [ ]:
# POOLS[config_id][problem_id] -> candidates, sorted by sample_index. One configuration is
# one temperature at one chain cap, which is what page 7's config_id already names.
POOLS, CONFIG_META = {}, {}
for r in raw:
    POOLS.setdefault(r["config_id"], {}).setdefault(r["problem_id"], []).append(r)
for cid in POOLS:
    for pid in POOLS[cid]:
        POOLS[cid][pid].sort(key=lambda r: r["sample_index"])

# A configuration is usable when enough items carry enough candidates. Trimming every item
# to the same depth is what makes an average across items one measurement.
usable = {}
for cid, items_at in POOLS.items():
    deep = {p: v for p, v in items_at.items() if len(v) >= MIN_POOL}
    if not deep:
        continue
    depth = min(len(v) for v in deep.values())
    usable[cid] = {p: v[:depth] for p, v in deep.items()}
    t = next(iter(deep.values()))[0]
    CONFIG_META[cid] = dict(config_id=cid, temperature=t["temperature"],
                            top_p=t.get("top_p"), chain_budget=t["chain_budget"],
                            pool_n=depth, n_items=len(deep))
POOLS = usable
assert POOLS, (f"no configuration carries {MIN_POOL} candidates on any item. Lower MIN_POOL, "
               f"or wait for the run that is making them to finish.")

# One chain length. Ranked by how much pile each cap carries, so "the deepest cap" means
# the one the page is actually about rather than whichever sorts first.
caps = Counter()
for cid, m in CONFIG_META.items():
    caps[m["chain_budget"]] += m["n_items"] * m["pool_n"]
CAP = CHAIN_CAP if CHAIN_CAP else max(caps, key=lambda c: (caps[c], c))
dropped_caps = sorted(c for c in caps if c != CAP)
POOLS = {c: v for c, v in POOLS.items() if CONFIG_META[c]["chain_budget"] == CAP}
assert POOLS, f"no configuration at cap {CAP}"

# Greedy is not a pile: N copies of one deterministic chain are one chain, and every rule
# here would score identically on it. It is kept out of the ladder and used in section 11
# as the reference the logprob diagnostic is compared against.
GREEDY_CONFIG = next((c for c in POOLS if CONFIG_META[c]["temperature"] == 0.0), None)
LADDER_CONFIGS = sorted((c for c in POOLS if c != GREEDY_CONFIG),
                        key=lambda c: CONFIG_META[c]["temperature"])
assert LADDER_CONFIGS, "every configuration on disk is greedy; there is no pile to select from"

# Two item sets, because a pilot pool is not square. A pool generated to answer page 7's
# question covers every item at its own temperature and a subset at the others, so:
#
#   ITEMS_AT[cid]  the items that configuration actually has. Figure 1 and the buckets are
#                  drawn on these, at their full strength.
#   GRID_ITEMS     the items EVERY configuration has. The temperature grid is drawn on
#                  these and on nothing else, because comparing temperatures across
#                  different problems would compare difficulty, not temperature.
ITEMS_AT = {c: sorted(POOLS[c]) for c in LADDER_CONFIGS}
GRID_ITEMS = sorted(set.intersection(*[set(v) for v in ITEMS_AT.values()]))
if ITEM_LIMIT:
    ITEMS_AT = {c: v[:ITEM_LIMIT] for c, v in ITEMS_AT.items()}
    GRID_ITEMS = [p for p in GRID_ITEMS if any(p in v for v in ITEMS_AT.values())][:ITEM_LIMIT]
    print(f"ITEM_LIMIT: {ITEM_LIMIT} items per configuration")
assert GRID_ITEMS, "the configurations share no items, so no temperature can be compared"

# The configuration the page's own figures are drawn on: the one carrying the most pile.
MAIN_CONFIG = max(LADDER_CONFIGS, key=lambda c: len(ITEMS_AT[c]) * CONFIG_META[c]["pool_n"])
ITEMS = ITEMS_AT[MAIN_CONFIG]

print(f"chain cap {CAP} tokens"
      + (f"   ({', '.join(str(c) for c in dropped_caps)} dropped: shorter chains are "
         f"page 7's axis, not this page's)" if dropped_caps else ""))
print(f"\n{'configuration':<16}{'T':>6}{'items':>7}{'pool':>6}   role")
for cid in sorted(CONFIG_META, key=lambda c: (CONFIG_META[c]["chain_budget"],
                                              CONFIG_META[c]["temperature"])):
    m = CONFIG_META[cid]
    if m["chain_budget"] != CAP:
        role = f"not used (cap {m['chain_budget']})"
    elif cid == GREEDY_CONFIG:
        role = "greedy reference, not a pile"
    else:
        role = "ladder" + (", and the page's own figures" if cid == MAIN_CONFIG else "")
    print(f"{cid:<16}{m['temperature']:>6.2f}{m['n_items']:>7}{m['pool_n']:>6}   {role}")
print(f"\n{len(GRID_ITEMS)} items shared by every configuration: the temperature grid")
print(f"{len(ITEMS)} items at {MAIN_CONFIG}: figure 1, the buckets and the worked case")
print(f"{sum(len(POOLS[c][p]) for c in LADDER_CONFIGS for p in ITEMS_AT[c]):,} candidates "
      f"in the ladder")
if len(GRID_ITEMS) < len(ITEMS):
    print(f"\nThe grid is the narrow end: {len(GRID_ITEMS)} items resolves a shape and does")
    print(f"not resolve a few points of accuracy. GENERATE fills the hot configurations out")
    print(f"to the full item set, which is most of what phase 1 is FOR.")


## 2. Page 7's vote, carried over unchanged

The vote is one of the three lines, so it is reimplemented here rather than approximated: the same
`number_only` vote key, the same equivalence relation, the same rule that a tie is not a vote. If it
drifts from page 7 the two pages report different numbers for one procedure, so section 13 checks it
against page 7's own `derived.json`.


In [ ]:
NO_ANSWER = None


def vote_key(text):
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ:
        try:
            _EQ[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ[k] = False
    return _EQ[k]


def cluster(keys):
    """Answer strings grouped into equivalence classes, in first-seen order."""
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


def plurality(cands):
    """The class the most candidates agree on, or None on a tie or an empty tally."""
    labels, _ = cluster([c.get("vote_key") or vote_key(c["trace"]) for c in cands])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    mc = counts.most_common()
    if not mc:
        return None
    top = mc[0][1]
    winners = [a for a, c in mc if c == top]
    return winners[0] if len(winners) == 1 else None


def vote_correct(cands):
    win = plurality(cands)
    return bool(win is not None and grade_answer(win, cands[0]["ground_truth"]))


## 3. The bar, measured before anything is scored

`bar(N) = majority(N) / coverage(N)`, the reliability a verifier must reach to draw with counting.

Two things to read off it. It **falls as N grows**, because coverage keeps climbing while the vote
saturates, so a wider pool is a cheaper win. And it **falls as temperature rises**, because
diversity puts more right answers in the pile and takes the crowd away from the vote. Those two
facts are the page's mechanism, and the whole point of measuring the bar here, before any verifier
exists, is that the threshold cannot then be chosen to suit the result.


In [ ]:
def n_axis_for(depth):
    ns = [n for n in (1, 2, 4, 8, 16, 32, 64, 128) if n <= depth]
    if ns[-1] != depth:
        ns.append(depth)
    return ns


def coverage_at(n_correct, pool_n, n):
    """P(at least one correct in a draw of n), exactly. The unbiased pass@N estimator,
    which is cheaper and less noisy than resampling for this one quantity."""
    if n_correct == 0:
        return 0.0
    if pool_n - n_correct < n:
        return 1.0
    return 1.0 - math.comb(pool_n - n_correct, n) / math.comb(pool_n, n)


def cluster_ci(per_item, reps=N_BOOTSTRAP, tag=""):
    """A 95% interval over ITEMS, which are the independent unit here.

    The draws inside one item are subsets of one pile and are heavily correlated, so a
    Wilson interval on the draw count would treat thousands of correlated draws as
    independent trials and come out several times too narrow.
    """
    vals = list(per_item)
    if not vals:
        return (0.0, 0.0)
    rng = random.Random(harness.sha256_of([str(SEED), "ci", tag, str(len(vals))])[:15])
    n, means = len(vals), []
    for _ in range(reps):
        means.append(sum(vals[rng.randrange(n)] for _ in range(n)) / n)
    means.sort()
    return (means[int(0.025 * reps)], means[int(0.975 * reps)])


def subsets_for(pid, cid, n, repeats, size):
    """The draws every selector sees at this (item, configuration, N). Exhaustive when the
    pile is small enough that resampling would just repeat the same subsets."""
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), cid, pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def bar_rows(cid, item_ids, tag=""):
    """majority, coverage and their ratio at every N, on one item set."""
    rows = []
    p1 = mean([mean([x["correct"] for x in POOLS[cid][p]]) for p in item_ids])
    for n in n_axis_for(CONFIG_META[cid]["pool_n"]):
        cov, maj = [], []
        for pid in item_ids:
            c = POOLS[cid][pid]
            cov.append(coverage_at(sum(x["correct"] for x in c), len(c), n))
            maj.append(mean([vote_correct([c[i] for i in idx])
                             for idx in subsets_for(pid, cid, n, N_RESAMPLES, len(c))]))
        m, cv = mean(maj), mean(cov)
        lo, hi = cluster_ci(maj, tag=f"bar-{tag}{cid}-{n}")
        clo, chi = cluster_ci(cov, tag=f"cov-{tag}{cid}-{n}")
        rows.append(dict(n=n, n_items=len(item_ids), pass_at_1=p1,
                         majority=m, majority_lo=lo, majority_hi=hi,
                         coverage=cv, coverage_lo=clo, coverage_hi=chi,
                         bar=(m / cv) if cv else float("nan"),
                         bar_lo=(lo / chi) if chi else float("nan"),
                         bar_hi=(hi / clo) if clo else float("nan")))
    return rows


# On each configuration's own items, which is what figure 1 is drawn from.
BAR = {cid: bar_rows(cid, ITEMS_AT[cid]) for cid in LADDER_CONFIGS}
# On the items every configuration shares, which is the only fair way to compare them.
BAR_GRID = {cid: bar_rows(cid, GRID_ITEMS, tag="grid-") for cid in LADDER_CONFIGS}

print("the bar: reliability a verifier needs to DRAW with counting\n")
for cid in LADDER_CONFIGS:
    m = CONFIG_META[cid]
    print(f"T={m['temperature']:.2f}, n={len(ITEMS_AT[cid])} items   {'N':>4}"
          f"{'one chain':>11}{'majority':>10}{'coverage':>10}{'bar':>8}")
    for r in BAR[cid]:
        print(f"{'':<24}{r['n']:>4}{r['pass_at_1']:>11.1%}{r['majority']:>10.1%}"
              f"{r['coverage']:>10.1%}{r['bar']:>8.0%}")
    print()

# "At its best" is read off each configuration's own items at its own deepest N, because
# that is the number each rule would actually be quoted at.
BEST_VOTE_CONFIG = max(LADDER_CONFIGS, key=lambda c: BAR[c][-1]["majority"])
LOWEST_BAR_CONFIG = min(LADDER_CONFIGS, key=lambda c: BAR[c][-1]["bar"])
bv, lb = BAR[BEST_VOTE_CONFIG][-1], BAR[LOWEST_BAR_CONFIG][-1]
print(f"counting is strongest at {BEST_VOTE_CONFIG}  "
      f"{bv['majority']:.0%} at N={bv['n']}, n={bv['n_items']}")
print(f"the bar is lowest at     {LOWEST_BAR_CONFIG}  "
      f"{lb['bar']:.0%} at N={lb['n']}, n={lb['n_items']} "
      f"[{lb['bar_lo']:.0%}-{lb['bar_hi']:.0%}]")
print("\nA verifier that clears the second number wins its own configuration. A verifier")
print("that beats the FIRST number wins the page, because that is counting at its best,")
print("and section 12 is where that comparison is made rather than assumed.")


# Part 2 — The verifier

The rung the page rests on. Everything above this point is free; everything below needs weights.


## 4. What a verifier is asked, and how the answer is read

The judge is put the candidate and asked one word, and the score is
`log P(Yes) - log P(No)` at the position where that word would start. One forward pass per
candidate: no sampling, no generation, and no verdict text to parse. Reading a free-text verdict
would fail exactly on the candidates where the model hedges, which are the interesting ones, and a
continuous score is what ranking needs anyway.

The judge sees the question and the candidate. It never sees the ground truth and never sees the
other candidates, which is what keeps it strictly below the oracle rung and what makes its score a
judgement rather than a vote.

**A trained reward model is scored by its own head instead**, with no prompt of ours in the way.
That is Cobbe et al.'s construction rather than a chat model imitating it, and it is the tier to
reach for if the chat judges fall short of the bar.

### Which checkpoints, and why size matters more than subject

The tiers are a **size ladder around the model that wrote the candidates**, not a hunt for the
best available judge, and that is deliberate.

A judge that is simply a stronger model at the task is a confound. If it can tell which candidate
is right, it can probably just solve the problem, and then "selection beat counting" and "a bigger
model beat a smaller one" are the same sentence. The deck's own numbers make this concrete: page 6
measured `Qwen2.5-7B-Instruct` at **60%** on these items at this cap, and the 3B's vote over 16
chains reaches **55%**. One chain from the 7B already beats the whole pile's vote, so a 7B judge
cannot carry this page on its own.

So the rung that settles the claim is the one **at or below the writer's size**.
`Qwen2.5-1.5B-Instruct` solves **22%** of these items against the 3B's 43%: it is worse at the
problems than the model whose work it is grading. If it still selects better than the crowd counts,
that is not "a bigger model helps", it is *verification is easier than generation*, which is the
thing best-of-N actually rests on. Section 12 reports that rung separately for exactly this reason.

**Why no math-tuned checkpoint.** Two reasons, both measured rather than assumed. Experiment 2 in
this repo found every `Qwen2.5-Math` model at **0% compliance** with a plain instruction, and a
one-word Yes/No verdict is precisely the instruction-following they are worst at. And their
accuracy on these items has never been measured here, while all three general checkpoints carry a
page 6 number, which is what lets the table above say what each judge's own competence is instead
of assuming it.


In [ ]:
JUDGE_PROMPT = (
    "You are grading a proposed solution to a mathematics problem.\n\n"
    "Problem:\n{question}\n\n"
    "Proposed solution:\n{solution}\n\n"
    "Is the final answer correct? Reply with one word: Yes or No."
)
_YES_NO = {}


def yes_no_ids(tok):
    """The single tokens a verdict would start with. Asserted single, because scoring the
    first token of a two-token word would score a prefix and not a verdict."""
    key = id(tok)
    if key not in _YES_NO:
        ids = {}
        for word in ("Yes", "No"):
            enc = tok(word, add_special_tokens=False).input_ids
            assert len(enc) == 1, (
                f"{word!r} is {len(enc)} tokens on this tokenizer, so the verdict cannot be "
                f"read off one position. Pick different verdict words and record them.")
            ids[word] = enc[0]
        _YES_NO[key] = ids
    return _YES_NO[key]


@torch.inference_mode()
def yes_no_score(model, tok, question, solution):
    """log P(Yes) - log P(No) at the verdict position. Positive means the judge believes
    the solution; the magnitude is what ranks candidates."""
    ids = yes_no_ids(tok)
    chat = tok.apply_chat_template(
        [{"role": "user", "content": JUDGE_PROMPT.format(question=question,
                                                         solution=solution)}],
        tokenize=False, add_generation_prompt=True)
    enc = tok(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
    logits = model(**enc).logits[0, -1].float()
    lp = torch.log_softmax(logits, dim=-1)
    return float(lp[ids["Yes"]] - lp[ids["No"]])


@torch.inference_mode()
def reward_score(model, tok, question, solution):
    """A trained reward model's own scalar, with no prompt of ours interposed. The chat
    template is the model card's, because a reward model is calibrated to the format it
    was trained on and reformatting it silently changes what is being measured."""
    chat = tok.apply_chat_template(
        [{"role": "user", "content": question}, {"role": "assistant", "content": solution}],
        tokenize=False)
    enc = tok(chat, return_tensors="pt", add_special_tokens=False,
              truncation=True, max_length=getattr(tok, "model_max_length", 4096)).to(model.device)
    out = model(**enc).logits[0]
    return float(out[-1] if out.ndim == 1 else out.flatten()[-1])


def load_verifier(spec):
    """The writer's own checkpoint is already resident, so `self` loads nothing."""
    if spec["kind"] == "rm":
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(spec["model"])
        kw = {}
        if spec.get("quant") == "8bit":
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        else:
            kw["dtype"] = harness.pick_dtype(get_device())
        mdl = AutoModelForSequenceClassification.from_pretrained(spec["model"], **kw).eval()
        if not kw.get("quantization_config"):
            mdl = mdl.to(get_device())
        return mdl, tok
    dev = get_device()
    got = harness.load_model(spec["model"], device=dev, dtype=harness.pick_dtype(dev),
                             load_in_8bit=(spec.get("quant") == "8bit"), warmup_tokens=8)
    return got["model"], got["tokenizer"]


SCORE_FN = {"yesno": yes_no_score, "rm": reward_score}
ACTIVE = [v for v in VERIFIERS if v["enabled"]]
for v in ACTIVE:
    if v["name"] != "self":
        assert v["model"], (
            f"verifier {v['name']!r} is enabled with no checkpoint named. Set `model` to one "
            f"verified against the hub, or disable the tier.")
WRITER_SOLVES = next((v["solves"] for v in VERIFIERS if v["name"] == "self"), None)
print(f"{len(ACTIVE)} verifier tier(s) enabled\n")
print(f"  {'tier':<11}{'kind':<8}{'solves these items alone':>26}   checkpoint")
for v in ACTIVE:
    mid = MODEL_ID if v["name"] == "self" else v["model"]
    solves = "unmeasured" if v.get("solves") is None else f"{v['solves']:.0%}"
    print(f"  {v['name']:<11}{v['kind']:<8}{solves:>26}   {mid.split('/')[-1]}")
print(f"\n  the pile's writer solves {WRITER_SOLVES:.0%} and its vote at the deepest N gets "
      f"{BAR[BEST_VOTE_CONFIG][-1]['majority']:.0%}")

OUTSOLVES = [v["name"] for v in ACTIVE
             if v.get("solves") is not None
             and v["solves"] > BAR[BEST_VOTE_CONFIG][-1]["majority"]]
if OUTSOLVES:
    print(f"\n  Read this before reading any curve. {', '.join(OUTSOLVES)} solves more of")
    print("  this item set on its own than the whole pile's vote does. Beating the vote")
    print("  with it is therefore ambiguous between 'selection beats counting' and 'a")
    print("  bigger model beats a smaller one', and the second is not this page's claim.")
    print("  The tiers that settle it are the ones at or below the writer's own size, and")
    print("  section 12 reports them separately for exactly that reason.")


## 5. The scoring pass

Every candidate, every enabled verifier, resumable and keyed so a re-run costs only what it adds.
`REPLAY` reads `scores.jsonl` and loads nothing.

The free scorers are recomputed in every mode: `heuristic_score` is pure text analysis, so replaying
a published number it could derive itself would be trusting where it could check.


In [ ]:
SCORE_ID = f"p08-{SOURCE_RUN_ID}"
SCORES = {}          # (config_id, problem_id, sample_index) -> {field: value}


def score_slot(cid, pid, si):
    return SCORES.setdefault((cid, pid, si), {})


# --- the free scorers, always recomputed -----------------------------------------
for cid in LADDER_CONFIGS:
    for pid in ITEMS_AT[cid]:
        for r in POOLS[cid][pid]:
            s = score_slot(cid, pid, r["sample_index"])
            s["heuristic"] = heuristic_score(r["trace"])
            s["n_chars"] = len(r["trace"])

# --- whatever a previous run left on disk ----------------------------------------
if SCORES_URL and not SCORES_PATH.exists():
    print(f"fetching published scores from {SCORES_URL}")
    SCORES_PATH.write_text(requests.get(SCORES_URL, timeout=600).text, encoding="utf-8")

read_back = 0
if SCORES_PATH.exists():
    for line in SCORES_PATH.open(encoding="utf-8"):
        if not line.strip():
            continue
        s = json.loads(line)
        if s.get("score_id") != SCORE_ID:
            continue
        k = (s["config_id"], s["problem_id"], s["sample_index"])
        if k in SCORES:
            SCORES[k].update({f: s[f] for f in s if f.startswith("v:")
                              or f in ("avg_logprob", "avg_logprob_chain")})
            read_back += 1
print(f"{read_back:,} scored candidates read from {SCORES_PATH}" if read_back
      else f"no scores on disk yet for {SCORE_ID}")


In [ ]:
def answer_span(trace):
    """(everything before the answer, the answer). Falls back to the whole trace when a
    candidate never opened a box, which is the case the vote calls 'no answer'."""
    i = trace.rfind("\\boxed{")
    return (trace[:i], trace[i:]) if i != -1 else ("", trace)


class _LogitsOnly:
    """A transformers model that answers like the book's model does."""

    def __init__(self, model):
        self.m = model

    def __call__(self, ids):
        return self.m(ids).logits


class _PlainEncode:
    """A tokenizer whose `.encode` adds nothing of its own. AutoTokenizer.encode adds
    special tokens by default, which would put a marker inside the scored sequence."""

    def __init__(self, tok):
        self.t = tok

    def encode(self, s):
        return self.t(s, add_special_tokens=False).input_ids


SCORED_THIS_RUN = Counter()

if RUN_MODE in ("SCORE", "GENERATE"):
    store = harness.TraceStore(SCORES_PATH, SCORE_ID,
                               ("config_id", "problem_id", "sample_index", "scorer"))
    store.report()
    work = [(cid, pid, r) for cid in LADDER_CONFIGS for pid in ITEMS_AT[cid]
            for r in POOLS[cid][pid]]

    for spec in ACTIVE:
        field = f"v:{spec['name']}"
        todo = [w for w in work if field not in SCORES[(w[0], w[1], w[2]["sample_index"])]]
        cached = [w for w in work if w not in todo]
        for cid, pid, r in cached:
            hit = store.get(cid, pid, r["sample_index"], spec["name"])
            if hit is not None:
                SCORES[(cid, pid, r["sample_index"])][field] = hit["score"]
        todo = [w for w in work
                if field not in SCORES[(w[0], w[1], w[2]["sample_index"])]]
        if not todo:
            print(f"{spec['name']}: already complete")
            continue

        model_id = MODEL_ID if spec["name"] == "self" else spec["model"]
        print(f"\n{spec['name']}: {len(todo):,} candidates to score with {model_id}")
        vmodel, vtok = load_verifier(dict(spec, model=model_id))
        fn = SCORE_FN[spec["kind"]]

        # The writer's checkpoint doubles as the logprob diagnostic, which needs the same
        # weights and would otherwise cost a second load of the same model.
        do_logprob = (spec["name"] == "self" and spec["kind"] == "yesno")
        if do_logprob:
            lp_model, lp_tok = _LogitsOnly(vmodel), _PlainEncode(vtok)

        bar = harness.progress(todo, desc=f"{spec['name']:<12}", unit="cand")
        for cid, pid, r in bar:
            k = (cid, pid, r["sample_index"])
            rec = {"score_id": SCORE_ID, "config_id": cid, "problem_id": pid,
                   "sample_index": r["sample_index"], "scorer": spec["name"],
                   "score": fn(vmodel, vtok, r["question"], r["trace"]),
                   "model": model_id, "quant": spec.get("quant"), "kind": spec["kind"]}
            if do_logprob:
                chat = vtok.apply_chat_template(
                    [{"role": "user", "content": render_prompt(r["question"])}],
                    tokenize=False, add_generation_prompt=True)
                head, tail = answer_span(r["trace"])
                rec["avg_logprob"] = (avg_logprob_answer(lp_model, lp_tok, chat + head, tail,
                                                         device=vmodel.device)
                                      if tail.strip() else None)
                SCORES[k]["avg_logprob"] = rec["avg_logprob"]
            store.append(rec)
            SCORES[k][field] = rec["score"]
            SCORED_THIS_RUN[spec["name"]] += 1
        del vmodel
        harness.free_memory()
        print(f"{spec['name']}: {SCORED_THIS_RUN[spec['name']]:,} scored")

# Which verifiers actually cover every candidate. A partial verifier is not plotted: a
# curve drawn from the subset it happened to reach is a different measurement.
HAVE = []
for spec in ACTIVE:
    field = f"v:{spec['name']}"
    n = sum(1 for cid in LADDER_CONFIGS for pid in ITEMS_AT[cid] for r in POOLS[cid][pid]
            if field in SCORES[(cid, pid, r["sample_index"])])
    total = sum(len(POOLS[c][p]) for c in LADDER_CONFIGS for p in ITEMS_AT[c])
    print(f"  {spec['name']:<10}{n:>8,} of {total:,} candidates scored"
          + ("   <- complete" if n == total else "   (incomplete, not plotted)"))
    if n == total:
        HAVE.append(spec)

HAVE_LOGPROB = all("avg_logprob" in SCORES[(c, p, r["sample_index"])]
                   for c in LADDER_CONFIGS for p in ITEMS_AT[c] for r in POOLS[c][p])
if not HAVE:
    print("\nNo verifier covers the pool, so the page has no best-of-N line yet. Set")
    print("RUN_MODE = 'SCORE' and run this notebook on a GPU: that is phase 0, it generates")
    print("nothing, and section 12 is the gate it decides.")


# Part 3 — What the page shows


## 6. The rules

Each takes a list of candidates and returns the index of the one it picks. That signature is the
page's whole argument: **same candidates, same compute, different answers.**

Ties among scores break toward the lowest sample index, which is arbitrary and stated rather than
hidden. The vote's ties are not broken at all, because page 7 does not break them.


In [ ]:
def sel_one_chain(cands, rng):
    """Plain CoT. One chain, chosen without looking at any of them, which is what a reader
    who runs the model once gets. Its curve is flat by construction, and that flatness is
    the thing the other two lines are measured against."""
    return rng.randrange(len(cands))


def sel_majority(cands, rng):
    """Page 7's rule. The index of a candidate carrying the winning answer, or None when
    the vote ties or nobody answered: a non-vote is not a pick."""
    win = plurality(cands)
    if win is None:
        return None
    for i, c in enumerate(cands):
        k = c.get("vote_key") or vote_key(c["trace"])
        if k is not NO_ANSWER and equivalent(k, win):
            return i
    return None


def sel_oracle(cands, rng):
    return next((i for i, c in enumerate(cands) if c["correct"]), None)


def by_score(field, ascending=False):
    """Highest score in `field` wins, ties to the lowest index. `ascending` exists for the
    heuristic, whose brevity term means its argmax is the shortest candidate."""
    sign = -1.0 if ascending else 1.0

    def pick(cands, rng):
        def key(i):
            s = SCORES[(cands[i]["config_id"], cands[i]["problem_id"],
                        cands[i]["sample_index"])].get(field)
            return (-math.inf if s is None else sign * s, -i)
        return max(range(len(cands)), key=key)
    return pick


# The three lines the page is about, plus the ceiling they share.
HEADLINE = [("one chain", "plain CoT: one sample, unexamined", sel_one_chain),
            ("majority", "keeps the answer that recurs", sel_majority)]
for spec in HAVE:
    HEADLINE.append((f"best-of-N ({spec['name']})",
                     f"keeps the candidate {spec['name']} scores highest",
                     by_score(f"v:{spec['name']}")))
HEADLINE.append(("coverage", "a correct answer is in the pile at all", sel_oracle))

# Diagnostics. Neither reads whether the reasoning is right, and section 11 is where that
# shows. They are kept off figure 1 because they are not what the page claims.
DIAGNOSTIC = [("heuristic", "ch05 heuristic_score: boxed, and short", by_score("heuristic"))]
if HAVE_LOGPROB:
    DIAGNOSTIC.append(("logprob", "ch05 avg_logprob_answer over the answer span",
                       by_score("avg_logprob")))

SELECTORS = HEADLINE + DIAGNOSTIC
print(f"{len(SELECTORS)} rules over the same candidates:")
for name, what, _ in SELECTORS:
    print(f"  {name:<22}{what}"
          + ("" if any(name == h[0] for h in HEADLINE) else "   (diagnostic)"))


In [ ]:
def run_ladder(cid, item_ids, pool_for, n_axis, selectors, tag=""):
    """Every selector over the same draws, at every N. `pool_for` returns the candidates an
    item contributes, so the buckets below can hand it a held-out half.

    Every rule sees the SAME subsets. Comparing rules across different draws would put
    sampling noise into the spread the figure reads as a difference between rules.
    """
    out = {name: [] for name, _, _ in selectors}
    picks = []
    for n in n_axis:
        per_item = {name: [] for name, _, _ in selectors}
        for pid in item_ids:
            cands_all = pool_for(pid)
            if not cands_all:
                continue
            hits = {name: [] for name, _, _ in selectors}
            for rep, idx in enumerate(
                    subsets_for(pid, cid + tag, n, N_RESAMPLES, len(cands_all))):
                cands = [cands_all[i] for i in idx]
                rng = random.Random(
                    harness.sha256_of([str(SEED), tag, cid, pid, str(n), str(rep)])[:15])
                for name, _, fn in selectors:
                    pick = fn(cands, rng)
                    ok = bool(pick is not None and cands[pick]["correct"])
                    hits[name].append(ok)
                    if n == n_axis[-1] and not tag:
                        picks.append(dict(config_id=cid, problem_id=pid, selector=name, n=n,
                                          pick=None if pick is None
                                          else cands[pick]["sample_index"], correct=ok))
            for name, _, _ in selectors:
                per_item[name].append(mean(hits[name]))
        for name, _, _ in selectors:
            vals = per_item[name]
            lo, hi = cluster_ci(vals, tag=f"{tag}{cid}{name}{n}")
            out[name].append(dict(n=n, accuracy=mean(vals), ci_lo=lo, ci_hi=hi,
                                  n_items=len(vals)))
    return out, picks


LADDERS, PICKS = {}, []
for cid in LADDER_CONFIGS:
    axis = n_axis_for(CONFIG_META[cid]["pool_n"])
    lad, pk = run_ladder(cid, ITEMS_AT[cid], lambda pid, c=cid: POOLS[c][pid], axis,
                         SELECTORS)
    LADDERS[cid] = dict(axis=axis, ladder=lad)
    PICKS.extend(pk)

# The same rules again on the items every configuration shares, at the depth every
# configuration reaches. Anything that compares one temperature to another reads THIS,
# because otherwise a temperature difference and a difference in which problems were
# asked would be the same number.
GRID_AXIS = n_axis_for(min(CONFIG_META[c]["pool_n"] for c in LADDER_CONFIGS))
GRID_N = GRID_AXIS[-1]
LADDERS_GRID = {}
for cid in LADDER_CONFIGS:
    lad, _ = run_ladder(cid, GRID_ITEMS, lambda pid, c=cid: POOLS[c][pid], GRID_AXIS,
                        HEADLINE, tag="grid-")
    LADDERS_GRID[cid] = dict(axis=GRID_AXIS, ladder=lad)
print(f"\npaired grid: every configuration re-measured on the same {len(GRID_ITEMS)} items "
      f"at N={GRID_N}")

for cid in LADDER_CONFIGS:
    print(f"\nT={CONFIG_META[cid]['temperature']:.2f}, n={len(ITEMS_AT[cid])}   {'N':>4}"
          + "".join(f"{name:>22}" for name, _, _ in SELECTORS))
    for i, n in enumerate(LADDERS[cid]["axis"]):
        print(f"        {n:>4}"
              + "".join(f"{LADDERS[cid]['ladder'][name][i]['accuracy']:>21.1%} "
                        for name, _, _ in SELECTORS))

# A consistency check that costs nothing: one chain drawn at random must land on the same
# number as the mean candidate, because it IS one candidate. If it does not, the draws are
# wrong and every curve above them is wrong with them.
for cid in LADDER_CONFIGS:
    single = mean([mean([c["correct"] for c in POOLS[cid][p]]) for p in ITEMS_AT[cid]])
    got = LADDERS[cid]["ladder"]["one chain"][0]["accuracy"]
    tol = max(0.03, 3 * 0.5 / max(len(ITEMS_AT[cid]) * N_RESAMPLES, 1) ** 0.5)
    assert abs(got - single) < tol, (
        f"{cid}: one chain at N=1 is {got:.1%} against a mean candidate of {single:.1%}; "
        f"the subsampling is not doing what it says")
print(f"\none chain at N=1 matches the mean candidate at every configuration "
      f"(tolerance {tol:.1%})")

# Having passed that check, the sampled version is replaced by the exact one. One chain
# does not depend on N -- it is pass@1 by definition -- and at the deepest N the pile has
# exactly one subset, so the sampled version is a single draw per item and visibly wobbles.
# A baseline that wobbles invites the reader to read a trend into a line that has none.
for lads, ids_for in ((LADDERS, lambda c: ITEMS_AT[c]), (LADDERS_GRID, lambda c: GRID_ITEMS)):
    for cid in LADDER_CONFIGS:
        per_item = [mean([x["correct"] for x in POOLS[cid][p]]) for p in ids_for(cid)]
        p1 = mean(per_item)
        lo, hi = cluster_ci(per_item, tag=f"p1-{cid}-{len(per_item)}")
        for pt in lads[cid]["ladder"]["one chain"]:
            pt.update(accuracy=p1, ci_lo=lo, ci_hi=hi, exact=True)
print("the one-chain line is drawn from pass@1 exactly, and is flat by construction")


## 7. Figure 1 — does judging beat counting

Accuracy against N at one configuration: plain CoT, the vote, best-of-N, and the coverage ceiling
they share. **Three lines and a ceiling, and nothing else**, because the page's question has three
answers and a seventh line on this chart is a way of not answering it.

The configuration drawn is the one where best-of-N is strongest. The rest of the grid is figure 2,
and section 12 checks this line against counting **at counting's own best temperature**, which is
the comparison that keeps the win honest.


In [ ]:
import matplotlib.pyplot as plt

INK = {"one chain": "#8a8578", "majority": "#3b4a6b", "coverage": "#4a7a5c"}
BON_INK = ["#b23", "#a2704a", "#7a4a8c"]
for i, spec in enumerate(HAVE):
    INK[f"best-of-N ({spec['name']})"] = BON_INK[i % len(BON_INK)]
for name in ("heuristic", "logprob"):
    INK.setdefault(name, "#c4c0b8")

BON_NAMES = [f"best-of-N ({s['name']})" for s in HAVE]


def best_of_n_at(cid, n=-1, ladders=None):
    """(name, point) for the strongest best-of-N line at a configuration."""
    if not BON_NAMES:
        return None, None
    L = (ladders or LADDERS)[cid]["ladder"]
    best = max(BON_NAMES, key=lambda nm: L[nm][n]["accuracy"])
    return best, L[best][n]


# Chosen on the paired grid, then drawn at full strength on its own items. Choosing it
# from the unpaired numbers would let a configuration win by having been asked easier
# problems.
# With no verifier there is no best-of-N to maximise, so the figures fall back to the
# configuration carrying the most items rather than to the hottest one: the diagnostics
# and the worked case are worth more where they are best resolved.
FIG1_CONFIG = (max(LADDER_CONFIGS,
                   key=lambda c: best_of_n_at(c, ladders=LADDERS_GRID)[1]["accuracy"])
               if BON_NAMES else MAIN_CONFIG)
print(f"figure 1 is drawn at {FIG1_CONFIG} "
      f"(n={len(ITEMS_AT[FIG1_CONFIG])}), chosen on the paired grid")

fig, ax = plt.subplots(figsize=(8.4, 4.8))
axis = LADDERS[FIG1_CONFIG]["axis"]
lad = LADDERS[FIG1_CONFIG]["ladder"]
for name, _, _ in HEADLINE:
    pts = lad[name]
    xs = [p["n"] for p in pts]
    ax.plot(xs, [p["accuracy"] * 100 for p in pts], marker="o", ms=4,
            lw=1.6 if name == "coverage" else 2.2,
            ls="--" if name == "coverage" else "-",
            color=INK[name], label=name, zorder=3)
    if name != "coverage":
        ax.fill_between(xs, [p["ci_lo"] * 100 for p in pts], [p["ci_hi"] * 100 for p in pts],
                        color=INK[name], alpha=0.10, lw=0)

if BON_NAMES:
    nm, top = best_of_n_at(FIG1_CONFIG)
    maj = lad["majority"][-1]
    cov = lad["coverage"][-1]
    x = axis[-1]
    ax.annotate("", xy=(x, top["accuracy"] * 100), xytext=(x, maj["accuracy"] * 100),
                arrowprops=dict(arrowstyle="<->", color="#333", lw=1))
    ax.text(x * 0.97, (top["accuracy"] + maj["accuracy"]) * 50,
            f"+{(top['accuracy'] - maj['accuracy']) * 100:.0f} pts\nover counting",
            ha="right", va="center", fontsize=8, color="#333")
    ax.annotate("", xy=(x, cov["accuracy"] * 100), xytext=(x, top["accuracy"] * 100),
                arrowprops=dict(arrowstyle="<->", color="#888", lw=1))
    ax.text(x * 0.97, (cov["accuracy"] + top["accuracy"]) * 50,
            f"{(cov['accuracy'] - top['accuracy']) * 100:.0f} pts\nstill in the pile",
            ha="right", va="center", fontsize=8, color="#888")

m = CONFIG_META[FIG1_CONFIG]
ax.set_xscale("log", base=2)
ax.set_xticks(axis)
ax.set_xticklabels([str(n) for n in axis])
ax.set_xlabel(f"candidates the rule may look at ({m['chain_budget']} tokens each)")
ax.set_ylabel("share of problems solved (%)")
ax.set_ylim(0, min(100, lad["coverage"][-1]["accuracy"] * 100 + 12))
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
ax.set_title(f"{MODEL_ID.split('/')[-1]}, T={m['temperature']}, "
             f"n={len(ITEMS_AT[FIG1_CONFIG])} problems, bands are 95% bootstrap intervals "
             f"over items", fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_1_payoff.png", dpi=160)
plt.show()

if not BON_NAMES:
    print("No verifier is scored, so figure 1 shows the two free rules and the ceiling.")
    print("The gap between the vote and the ceiling is what a verifier is being asked for.")


## 8. Figure 2 — why it wins, and where

Temperature is the diversity knob, and the two rules spend diversity in opposite directions.
Coverage rises with it: more distinct attempts means the right answer is in the pile more often.
Consensus falls with it: there is less of a crowd to count. So the bar a verifier has to clear,
`majority / coverage`, falls as the pile gets more diverse.

**This is the page's mechanism**, and it is the reason best-of-N and majority voting are not two
tools for the same job. The shaded distance between the two lines is what selection has to work
with and counting does not.


In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.6, 4.2),
                               gridspec_kw=dict(width_ratios=[1.25, 1]))

Ts = [CONFIG_META[c]["temperature"] for c in LADDER_CONFIGS]


def at_n(cid, key, n=None, table=None):
    """One entry from the bar table. Defaults to the PAIRED table at the paired depth,
    because every cross-temperature statement on this page has to be paired."""
    rows = (table or BAR_GRID)[cid]
    return next(r[key] for r in rows if r["n"] == (n or GRID_N))


cov = [at_n(c, "coverage") * 100 for c in LADDER_CONFIGS]
maj = [at_n(c, "majority") * 100 for c in LADDER_CONFIGS]
axL.fill_between(Ts, maj, cov, color="#b23", alpha=0.10, lw=0,
                 label="what selection has to work with")
axL.plot(Ts, cov, marker="o", ms=4, color="#4a7a5c", ls="--", label="coverage")
axL.plot(Ts, maj, marker="o", ms=4, color="#3b4a6b", lw=2.2, label="majority vote")
axL.plot(Ts, [at_n(c, "pass_at_1") * 100 for c in LADDER_CONFIGS], marker="o", ms=3,
         color="#8a8578", lw=1.4, label="one chain")
if BON_NAMES:
    axL.plot(Ts, [best_of_n_at(c, ladders=LADDERS_GRID)[1]["accuracy"] * 100
                  for c in LADDER_CONFIGS],
             marker="o", ms=4, color=INK[BON_NAMES[0]], lw=2.2, label="best-of-N")
axL.set_xlabel("sampling temperature")
axL.set_ylabel(f"share of problems solved at N={GRID_N} (%)")
axL.set_xticks(Ts)
axL.set_ylim(0, 100)
axL.grid(alpha=0.22)
axL.spines[["top", "right"]].set_visible(False)
axL.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
axL.set_title("diversity feeds selection and starves counting", fontsize=9, loc="left",
              color="#555")

bars = [at_n(c, "bar") * 100 for c in LADDER_CONFIGS]
axR.fill_between(Ts, [at_n(c, "bar_lo") * 100 for c in LADDER_CONFIGS],
                 [min(at_n(c, "bar_hi") * 100, 105) for c in LADDER_CONFIGS],
                 color="#b23", alpha=0.10, lw=0)
axR.plot(Ts, bars, marker="o", ms=5, color="#b23", lw=2.2)
for t, bb in zip(Ts, bars):
    axR.annotate(f"{bb:.0f}%", (t, bb), textcoords="offset points", xytext=(0, 8),
                 ha="center", fontsize=8, color="#b23")
axR.set_xlabel("sampling temperature")
axR.set_ylabel("verifier reliability needed to draw with counting (%)")
axR.set_ylim(0, 105)
axR.grid(alpha=0.22)
axR.spines[["top", "right"]].set_visible(False)
axR.set_xticks(Ts)
axR.set_title(f"the bar, at N={GRID_N}, band is the interval on the ratio", fontsize=9,
              loc="left", color="#555")
fig.tight_layout()
fig.suptitle("")
fig.savefig(OUT_DIR / "figure_2_mechanism.png", dpi=160)
plt.show()

print(f"the same {len(GRID_ITEMS)} problems at every temperature, N={GRID_N}\n")
print(f"{'T':>6}{'one chain':>11}{'majority':>10}{'coverage':>10}{'bar':>8}"
      + ("".join(f"{'bo-N ' + v['name']:>14}" for v in HAVE)))
for cid in LADDER_CONFIGS:
    row = (f"{CONFIG_META[cid]['temperature']:>6.2f}{at_n(cid, 'pass_at_1'):>11.1%}"
           f"{at_n(cid, 'majority'):>10.1%}{at_n(cid, 'coverage'):>10.1%}"
           f"{at_n(cid, 'bar'):>8.0%}")
    for v in HAVE:
        nm = f"best-of-N ({v['name']})"
        i = GRID_AXIS.index(GRID_N)
        row += f"{LADDERS_GRID[cid]['ladder'][nm][i]['accuracy']:>14.1%}"
    print(row)
if len(GRID_ITEMS) < 60:
    print(f"\n{len(GRID_ITEMS)} items resolves the SHAPE of these lines and does not resolve")
    print("a few points of accuracy. Read the divergence, not the gaps.")


## 9. The three buckets, and the slice where selection is the only thing that can help

Every item falls into one of three, by what the vote does with it:

- **solvable** — the modal answer is correct. Page 7's regime, and counting works.
- **trap** — the modal answer is wrong and a correct answer appears at least once. The only place
  selection can beat counting.
- **hopeless** — no candidate is correct. The coverage ceiling, where no rule helps and only more
  generation or a better model can.

**The bucket is named on one half of the pile and measured on the other.** Defining the trap bucket
on the chains you then measure would guarantee that the vote looks bad there, because that is how
the bucket was chosen.


In [ ]:
# The buckets are pinned to the configuration with the most items, not to figure 1's.
# The base rate is the page's most quotable number and page 7 closes on the same one, so
# it is measured where it is best resolved.
BUCKET_CONFIG = MAIN_CONFIG
POOL_B = POOLS[BUCKET_CONFIG]
NAMING = {p: POOL_B[p][0::2] if BUCKET_HOLDOUT else POOL_B[p] for p in ITEMS}
MEASURING = {p: POOL_B[p][1::2] if BUCKET_HOLDOUT else POOL_B[p] for p in ITEMS}
BUCKET_N = min(len(v) for v in MEASURING.values())
BUCKET_AXIS = [n for n in n_axis_for(BUCKET_N)]


def bucket_of(cands):
    if vote_correct(cands):
        return "solvable"
    return "trap" if any(c["correct"] for c in cands) else "hopeless"


BUCKET = {p: bucket_of(NAMING[p]) for p in ITEMS}
BASE_RATE = {b: mean([BUCKET[p] == b for p in ITEMS])
             for b in ("solvable", "trap", "hopeless")}
FULL_PILE_RATE = {b: mean([bucket_of(POOL_B[p]) == b for p in ITEMS])
                  for b in ("solvable", "trap", "hopeless")}

print(f"at {BUCKET_CONFIG}: named by {len(next(iter(NAMING.values())))} chains, measured on "
      f"{BUCKET_N}, n={len(ITEMS)} problems")
for b, share in BASE_RATE.items():
    lo, hi = harness.wilson(round(share * len(ITEMS)), len(ITEMS))
    print(f"  {b:<10}{share:>6.0%}  {lo:.0%}-{hi:.0%}   {'#' * round(share * 34)}")
print("\n  the same three over the whole pile, which is what defining and measuring on one")
print("  set of chains would have reported: "
      + ", ".join(f"{b} {v:.0%}" for b, v in FULL_PILE_RATE.items()))
print("\n  trap = 'the right answer was there and the vote threw it away', which is what")
print("  page 7 closes on. The two numbers above are not interchangeable: page 7 quotes")
print(f"  the whole pile ({FULL_PILE_RATE['trap']:.0%}), and the held-out estimate "
      f"({BASE_RATE['trap']:.0%}) is lower because")
print("  each half is half as deep, so a correct answer is in it less often. The page")
print("  should quote the held-out number for the bucket CURVES, because that is what")
print("  they were measured on, and page 7's for the base rate it shares with page 7.")

TRAP_OK = BASE_RATE["trap"] >= 0.08
if not TRAP_OK:
    print(f"\n!! the trap bucket is {BASE_RATE['trap']:.0%}, too thin to plot a curve on")

BUCKET_LADDER = {}
for b in ("solvable", "trap", "hopeless"):
    ids = [p for p in ITEMS if BUCKET[p] == b]
    if not ids:
        continue
    lad, _ = run_ladder(BUCKET_CONFIG, ids, lambda pid: MEASURING[pid], BUCKET_AXIS,
                        HEADLINE, tag=f"bucket-{b}-")
    per_item = [mean([x["correct"] for x in MEASURING[p]]) for p in ids]
    lo, hi = cluster_ci(per_item, tag=f"p1-bucket-{b}")
    for pt in lad["one chain"]:
        pt.update(accuracy=mean(per_item), ci_lo=lo, ci_hi=hi, exact=True)
    BUCKET_LADDER[b] = dict(n_items=len(ids), share=len(ids) / len(ITEMS), ladder=lad)

for b, d in BUCKET_LADDER.items():
    print(f"\n{b}  ({d['n_items']} items, {d['share']:.0%})")
    print(f"  {'N':>4}" + "".join(f"{name:>22}" for name, _, _ in HEADLINE))
    for i, n in enumerate(BUCKET_AXIS):
        print(f"  {n:>4}" + "".join(f"{d['ladder'][name][i]['accuracy']:>21.1%} "
                                    for name, _, _ in HEADLINE))


In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.6, 3.8),
                               gridspec_kw=dict(width_ratios=[1, 1.15]))
left = 0.0
for b, colour in (("solvable", "#3b4a6b"), ("trap", "#b23"), ("hopeless", "#c4c0b8")):
    ax0.barh([0], [BASE_RATE[b] * 100], left=left * 100, color=colour, height=0.45)
    if BASE_RATE[b] > 0.06:
        ax0.text((left + BASE_RATE[b] / 2) * 100, 0, f"{b}\n{BASE_RATE[b]:.0%}",
                 ha="center", va="center", fontsize=9,
                 color="#fff" if b != "hopeless" else "#333")
    left += BASE_RATE[b]
ax0.set_xlim(0, 100)
ax0.set_ylim(-0.6, 0.6)
ax0.set_yticks([])
ax0.set_xlabel("share of problems (%)")
ax0.set_title(f"what the vote does, over {len(ITEMS)} problems", fontsize=9, loc="left",
              color="#555")
ax0.spines[["top", "right", "left"]].set_visible(False)

if "trap" in BUCKET_LADDER:
    d = BUCKET_LADDER["trap"]
    for name, _, _ in HEADLINE:
        pts = d["ladder"][name]
        xs = [p["n"] for p in pts]
        ax1.plot(xs, [p["accuracy"] * 100 for p in pts], marker="o", ms=4,
                 lw=1.6 if name == "coverage" else 2.2,
                 ls="--" if name == "coverage" else "-", color=INK[name], label=name)
        if name == "majority":
            ax1.fill_between(xs, [p["ci_lo"] * 100 for p in pts],
                             [p["ci_hi"] * 100 for p in pts], color=INK[name],
                             alpha=0.12, lw=0)
    ax1.set_xscale("log", base=2)
    ax1.set_xticks(BUCKET_AXIS)
    ax1.set_xticklabels([str(n) for n in BUCKET_AXIS])
    ax1.set_xlabel("candidates the rule may look at")
    ax1.set_ylabel("share of the trap slice solved (%)")
    ax1.set_ylim(0, 100)
    ax1.grid(axis="y", alpha=0.25)
    ax1.legend(frameon=False, fontsize=8, loc="upper left")
    ax1.set_title(f"the trap slice: {d['n_items']} problems where the vote is wrong and a "
                  f"correct answer is in the pile", fontsize=9, loc="left", color="#555")
else:
    ax1.text(0.5, 0.5, "no trap items", ha="center", va="center")
ax1.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2b_buckets.png", dpi=160)
plt.show()

MAJORITY_DECLINES_ON_TRAPS = None
if "trap" in BUCKET_LADDER and len(BUCKET_AXIS) > 1:
    lad = BUCKET_LADDER["trap"]["ladder"]["majority"]
    MAJORITY_DECLINES_ON_TRAPS = bool(lad[-1]["accuracy"] < lad[0]["accuracy"]
                                      and lad[-1]["ci_hi"] < lad[0]["ci_lo"])
    print(f"on the trap slice the vote goes {lad[0]['accuracy']:.0%} -> "
          f"{lad[-1]['accuracy']:.0%}: "
          + ("declines, intervals separated" if MAJORITY_DECLINES_ON_TRAPS
             else "not resolved, the intervals overlap"))


## 10. Figure 3 — the verifier is worth exactly what the verifier is worth

**Reliability: on items where a correct candidate exists, how often does this rule pick one?** One
number per rule, on the same candidates, and it is the honest replacement for a slider the reader
drags.

The majority reference line is drawn across the scatter, and where it crosses is the measured
threshold: below it, judging is worse than counting and page 7's tool keeps its place in the
toolbox.

**The right panel is what the judge ablation is for.** Its x-axis is not the judge's reliability
but what the judge can do *on its own*, measured by page 6 on these same items. Every tier is a
point, and the question the panel answers is the one a reader actually has: **how good does the
judge have to be?**

The two axes are the page's argument in one picture. Accuracy is `reliability x coverage`, so the
judge gates how much of the pile is collectable, and the left panel is that identity drawn. What
the right panel adds is the price of the gate. A judge sitting far to the left of the writer's own
accuracy, still landing above the majority line, is *verification is easier than generation*,
demonstrated rather than cited, and the horizontal distance from the dotted line is how much
easier.


In [ ]:
FIG3_CONFIG = FIG1_CONFIG
reachable = {p for p in ITEMS_AT[FIG3_CONFIG]
             if any(c["correct"] for c in POOLS[FIG3_CONFIG][p])}
RELIABILITY = []
for name, what, _ in SELECTORS:
    picks = [q for q in PICKS if q["selector"] == name and q["config_id"] == FIG3_CONFIG
             and q["problem_id"] in reachable]
    i = -1
    RELIABILITY.append(dict(
        selector=name, what=what,
        reliability=mean([q["correct"] for q in picks]) if picks else 0.0,
        accuracy=LADDERS[FIG3_CONFIG]["ladder"][name][i]["accuracy"],
        n=LADDERS[FIG3_CONFIG]["axis"][i], headline=any(name == h[0] for h in HEADLINE)))
RELIABILITY_BY = {r["selector"]: r for r in RELIABILITY}

maj = RELIABILITY_BY["majority"]
bar_here = at_n(FIG3_CONFIG, "bar", LADDERS[FIG3_CONFIG]["axis"][-1])
print(f"at {FIG3_CONFIG}, on the {len(reachable)} of {len(ITEMS_AT[FIG3_CONFIG])} problems "
      f"where a correct candidate exists:")
print(f"  {'rule':<22}{'reliability':>13}{'accuracy':>11}")
for r in RELIABILITY:
    print(f"  {r['selector']:<22}{r['reliability']:>13.0%}{r['accuracy']:>11.0%}"
          + ("" if r["headline"] else "   (diagnostic)"))
print(f"\n  the bar here is {bar_here:.0%}: a rule above it beats counting, one below it "
      f"does not")

JUDGE_LADDER = [
    dict(name=v["name"], solves=v.get("solves"),
         params=None if v["name"] == "self" else v["model"],
         reliability=RELIABILITY_BY[f"best-of-N ({v['name']})"]["reliability"],
         accuracy=RELIABILITY_BY[f"best-of-N ({v['name']})"]["accuracy"])
    for v in HAVE]

fig, (ax, axB) = plt.subplots(1, 2, figsize=(11.2, 4.4),
                              gridspec_kw=dict(width_ratios=[1, 1]))
for r in RELIABILITY:
    if r["selector"] == "coverage":
        continue
    ax.scatter(r["reliability"] * 100, r["accuracy"] * 100, s=58 if r["headline"] else 34,
               color=INK.get(r["selector"], "#999"),
               marker="o" if r["headline"] else "^", zorder=3)
    ax.annotate(r["selector"], (r["reliability"] * 100, r["accuracy"] * 100),
                textcoords="offset points", xytext=(8, -3), fontsize=8, color="#333")
ax.axhline(maj["accuracy"] * 100, ls="--", lw=1, color="#3b4a6b")
ax.text(1.5, maj["accuracy"] * 100 + 1.4, "majority vote", fontsize=8, color="#3b4a6b")
ax.axvline(bar_here * 100, ls=":", lw=1, color="#b23")
ax.text(bar_here * 100 + 1.5, 3, f"the bar, {bar_here:.0%}", fontsize=8, color="#b23",
        rotation=90, va="bottom")
ax.set_xlabel("measured reliability: picks a correct candidate when one exists (%)")
ax.set_ylabel(f"accuracy at N={LADDERS[FIG3_CONFIG]['axis'][-1]} (%)")
ax.set_xlim(0, 105)
ax.set_ylim(0, min(100, max(r["accuracy"] for r in RELIABILITY) * 100 + 10))
ax.grid(alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"the gate: accuracy = reliability x coverage", fontsize=9, loc="left",
             color="#555")

# The second panel is the ablation's real deliverable. The x-axis is what each judge can
# do on its OWN, measured by page 6 on these same items, and the y-axis is what it unlocks
# as a selector. A judge far to the left that still lands above the majority line is the
# page's whole point: verification is cheaper than generation, and the gap between the two
# axes is how much cheaper.
plotted = [j for j in JUDGE_LADDER if j["solves"] is not None]
if len(plotted) >= 2:
    xs = [j["solves"] * 100 for j in plotted]
    ys = [j["accuracy"] * 100 for j in plotted]
    axB.plot(xs, ys, color="#b23", lw=1.4, ls="-", zorder=2, alpha=0.6)
    for j in plotted:
        axB.scatter(j["solves"] * 100, j["accuracy"] * 100, s=58,
                    color=INK.get(f"best-of-N ({j['name']})", "#b23"), zorder=3)
        axB.annotate(j["name"], (j["solves"] * 100, j["accuracy"] * 100),
                     textcoords="offset points", xytext=(8, -3), fontsize=8, color="#333")
    axB.axhline(maj["accuracy"] * 100, ls="--", lw=1, color="#3b4a6b")
    axB.text(1.5, maj["accuracy"] * 100 + 1.4, "majority vote", fontsize=8, color="#3b4a6b")
    if WRITER_SOLVES:
        axB.axvline(WRITER_SOLVES * 100, ls=":", lw=1, color="#555")
        axB.text(WRITER_SOLVES * 100 + 1.5, 3, "the writer's own accuracy", fontsize=8,
                 color="#555", rotation=90, va="bottom")
        axB.axvspan(WRITER_SOLVES * 100, 105, color="#555", alpha=0.05, lw=0)
    axB.set_xlim(0, 105)
    axB.set_ylim(0, min(100, max(ys + [maj["accuracy"] * 100]) + 12))
    axB.set_xlabel("what the judge solves on its own (%)")
    axB.set_ylabel(f"best-of-{LADDERS[FIG3_CONFIG]['axis'][-1]} accuracy it unlocks (%)")
    axB.grid(alpha=0.2)
    axB.set_title("how good does the judge have to be", fontsize=9, loc="left", color="#555")
else:
    axB.text(0.5, 0.5, "needs two or more scored judges\nwith a measured solve rate",
             ha="center", va="center", fontsize=9, color="#888")
    axB.set_xticks([])
    axB.set_yticks([])
axB.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_3_reliability.png", dpi=160)
plt.show()

CHEAPEST_SUFFICIENT = None
if JUDGE_LADDER:
    print(f"\n  {'judge':<11}{'solves alone':>14}{'reliability':>13}"
          f"{'accuracy it unlocks':>21}{'beats the vote':>16}")
    for j in sorted(JUDGE_LADDER, key=lambda j: (j["solves"] is None, j["solves"] or 0)):
        beats = j["accuracy"] > maj["accuracy"]
        print(f"  {j['name']:<11}"
              f"{('-' if j['solves'] is None else f'{j['solves']:.0%}'):>14}"
              f"{j['reliability']:>13.0%}{j['accuracy']:>21.0%}"
              f"{('yes' if beats else 'no'):>16}")
    winners = [j for j in JUDGE_LADDER if j["accuracy"] > maj["accuracy"]
               and j["solves"] is not None]
    if winners:
        CHEAPEST_SUFFICIENT = min(winners, key=lambda j: j["solves"])
        j = CHEAPEST_SUFFICIENT
        print(f"\n  The cheapest judge that beats counting solves {j['solves']:.0%} of these "
              f"problems itself,")
        print(f"  against the {WRITER_SOLVES:.0%} of the model whose work it is grading. It "
              f"picks a correct")
        print(f"  candidate {j['reliability']:.0%} of the time one is there, which is "
              f"{j['accuracy']:.0%} accuracy overall.")
        if j["solves"] < WRITER_SOLVES:
            print("\n  That is the page's sentence, and it is measured: a model that is WORSE")
            print("  at these problems than the one that wrote the answers can still tell")
            print("  which answer is right. Verification is easier than generation.")
        else:
            print("\n  Note that this judge is at least as good at the problems as the writer,")
            print("  so on this instrument the page cannot yet claim that verification is the")
            print("  easier job. It can claim that a judge of this quality is what it takes.")


## 11. What the diagnostics are actually ranking by

Before either ch05 scorer is called a verifier, it is worth measuring what it does, because both
turn out to be proxies for something plainer than their names suggest. This section is the reason
they are diagnostics and not the payoff.

Two numbers per scorer. **AUC** is how well the score separates a correct candidate from a wrong one
across the whole pool. **Reliability** is how often its argmax over N candidates lands on a correct
one. They can disagree sharply, and when they do it is because an argmax samples the tail of the
score distribution, where a mediocre signal is at its worst.


In [ ]:
import bisect


def auc(pairs):
    """P(score of a correct candidate > score of a wrong one), ties counted as half."""
    pos = [s for s, y in pairs if y and s is not None]
    neg = sorted(s for s, y in pairs if not y and s is not None)
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for s in pos:
        lo = bisect.bisect_left(neg, s)
        hi = bisect.bisect_right(neg, s)
        wins += lo + 0.5 * (hi - lo)
    return wins / (len(pos) * len(neg))


cands = [(c, SCORES[(FIG3_CONFIG, p, c["sample_index"])])
         for p in ITEMS_AT[FIG3_CONFIG] for c in POOLS[FIG3_CONFIG][p]]

# Cluster size is the vote's own per-candidate signal, and it is the number every verifier
# is really competing with: it is free, and on a low-temperature pile it is strong.
csize = {}
for p in ITEMS_AT[FIG3_CONFIG]:
    c = POOLS[FIG3_CONFIG][p]
    labels, _ = cluster([x.get("vote_key") or vote_key(x["trace"]) for x in c])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    for x, l in zip(c, labels):
        csize[(p, x["sample_index"])] = counts.get(l, 0)

SIGNALS = {"cluster size (the vote's own signal)":
           lambda c, s: csize[(c["problem_id"], c["sample_index"])],
           "ch05 heuristic (boxed, and short)": lambda c, s: s["heuristic"],
           "trace length": lambda c, s: s["n_chars"]}
if HAVE_LOGPROB:
    SIGNALS["ch05 avg_logprob (answer span)"] = lambda c, s: s.get("avg_logprob")
for spec in HAVE:
    SIGNALS[f"verifier: {spec['name']}"] = (
        lambda c, s, f=f"v:{spec['name']}": s.get(f))

DISCRIMINATION = []
print(f"over {len(cands):,} candidates at {FIG3_CONFIG}   (AUC 0.5 = no signal)\n")
print(f"  {'signal':<38}{'AUC':>7}{'reliability at argmax':>24}")
for label, fn in SIGNALS.items():
    a = auc([(fn(c, s), c["correct"]) for c, s in cands])
    rel = None
    for r in RELIABILITY:
        if (label.startswith("verifier: ") and r["selector"] == f"best-of-N ({label[10:]})") \
                or (label.startswith("ch05 heuristic") and r["selector"] == "heuristic") \
                or (label.startswith("ch05 avg_logprob") and r["selector"] == "logprob") \
                or (label.startswith("cluster size") and r["selector"] == "majority"):
            rel = r["reliability"]
    DISCRIMINATION.append(dict(signal=label, auc=a, reliability=rel))
    print(f"  {label:<38}{a:>7.3f}{(f'{rel:.0%}' if rel is not None else '-'):>24}")

HEURISTIC_IS_LENGTH = mean(["\\boxed{" in c["trace"] for c, _ in cands]) > 0.95
if HEURISTIC_IS_LENGTH:
    print("\n  Every candidate carries a box, so the heuristic's boxed bonus cancels and the")
    print("  rule ranks by its brevity term alone. On this pile 'heuristic score' means")
    print("  SHORTEST WINS and nothing else, which is a sharper thing to say on the page")
    print("  than 'it grades presentation'.")

GREEDY_AGREEMENT = None
if GREEDY_CONFIG and HAVE_LOGPROB:
    shared = [p for p in ITEMS_AT[FIG3_CONFIG] if p in POOLS[GREEDY_CONFIG]]
    if shared:
        pick = by_score("avg_logprob")
        agree = []
        for p in shared:
            c = POOLS[FIG3_CONFIG][p]
            g = POOLS[GREEDY_CONFIG][p][0]
            gk = g.get("vote_key") or vote_key(g["trace"])
            pk = c[pick(c, None)]
            pk = pk.get("vote_key") or vote_key(pk["trace"])
            agree.append(bool(gk and pk and equivalent(gk, pk)))
        GREEDY_AGREEMENT = dict(
            config_id=GREEDY_CONFIG, n_items=len(shared), answer_agreement=mean(agree),
            greedy_accuracy=mean([POOLS[GREEDY_CONFIG][p][0]["correct"] for p in shared]))
        print(f"\n  the logprob pick gives the same answer as the greedy chain on "
              f"{GREEDY_AGREEMENT['answer_agreement']:.0%} of items, and greedy alone solves "
              f"{GREEDY_AGREEMENT['greedy_accuracy']:.0%}")
        if GREEDY_AGREEMENT["answer_agreement"] > 0.8:
            print("  At that rate the logprob rung is largely recovering the greedy chain, so")
            print("  any gain it shows is mostly 'greedy beats a sample' rather than 'the")
            print("  model can tell which of its answers is right'. The page may not call it")
            print("  a confidence signal without saying this.")


## 12. The gate

**Does best-of-N beat counting, and beat one chain?**

The comparison is deliberately unkind to this page: best-of-N at *its* best configuration against
majority voting at *counting's* best configuration. Beating the vote only where the vote has been
starved of a crowd would be a rigged win, and this cell is what stops the page claiming one. The
whole temperature grid is printed above it, so a reader can see what was chosen from.


In [ ]:
def interval(cid, name, ladders=None, n=-1):
    p = (ladders or LADDERS)[cid]["ladder"][name][n]
    axis = (ladders or LADDERS)[cid]["axis"]
    return dict(config_id=cid, selector=name, n=axis[n], accuracy=p["accuracy"],
                ci=[p["ci_lo"], p["ci_hi"]])


# Counting at its literal best: its strongest configuration, at that configuration's
# deepest N, on that configuration's own items. This is the hard opponent, and picking a
# softer one is exactly the way this page could fool itself.
maj = interval(BEST_VOTE_CONFIG, "majority")
cot = max((dict(config_id=c, accuracy=BAR[c][0]["pass_at_1"]) for c in LADDER_CONFIGS),
          key=lambda d: d["accuracy"])
GATE = dict(majority_at_its_best=maj, one_chain_at_its_best=cot, best_of_n=None,
            paired=None)

print(f"counting at its best    {maj['config_id']}  N={maj['n']}  "
      f"n={len(ITEMS_AT[maj['config_id']])}   {maj['accuracy']:.1%} "
      f"[{maj['ci'][0]:.1%}-{maj['ci'][1]:.1%}]")
print(f"one chain at its best   {cot['config_id']}"
      f"{'':>{max(0, len(maj['config_id']) - len(cot['config_id']))}}"
      f"                    {cot['accuracy']:.1%}")

BEATS_COUNTING = BEATS_COT = False
if BON_NAMES:
    cfg, nm = max(((c, nm) for c in LADDER_CONFIGS for nm in BON_NAMES),
                  key=lambda k: LADDERS[k[0]]["ladder"][k[1]][-1]["accuracy"])
    bon = interval(cfg, nm)
    bon["verifier"] = nm[nm.find("(") + 1:-1]
    GATE["best_of_n"] = bon
    BEATS_COUNTING = bool(bon["accuracy"] > maj["accuracy"] and bon["ci"][0] > maj["ci"][1])
    BEATS_COT = bool(bon["accuracy"] > cot["accuracy"])
    print(f"best-of-N at its best   {cfg}  N={bon['n']}  n={len(ITEMS_AT[cfg])}   "
          f"{bon['accuracy']:.1%} [{bon['ci'][0]:.1%}-{bon['ci'][1]:.1%}]   ({nm})")

    # The same comparison restricted to verifiers that are NOT stronger generators than
    # the model that wrote the pile. This is the number the page's title can honestly
    # quote, and it is reported whether or not it is the best one.
    clean_names = [f"best-of-N ({v['name']})" for v in HAVE
                   if v.get("solves") is None or WRITER_SOLVES is None
                   or v["solves"] <= WRITER_SOLVES]
    if clean_names:
        ccfg, cnm = max(((c, n2) for c in LADDER_CONFIGS for n2 in clean_names),
                        key=lambda k: LADDERS[k[0]]["ladder"][k[1]][-1]["accuracy"])
        clean = interval(ccfg, cnm)
        GATE["best_of_n_no_stronger_generator"] = clean
        BEATS_COUNTING_CLEAN = bool(clean["accuracy"] > maj["accuracy"]
                                    and clean["ci"][0] > maj["ci"][1])
        print(f"  ... of which the best verifier that is NOT a stronger generator than the "
              f"writer:\n      {ccfg}  N={clean['n']}   {clean['accuracy']:.1%} "
              f"[{clean['ci'][0]:.1%}-{clean['ci'][1]:.1%}]   ({cnm})   "
              + ("CLEARS" if BEATS_COUNTING_CLEAN else "does not clear"))
    else:
        BEATS_COUNTING_CLEAN = False
        print("  ... and every scored verifier is a stronger generator than the writer, so")
        print("      none of them settles the page's claim on its own.")
    print(f"\n  vs counting  {(bon['accuracy'] - maj['accuracy']) * 100:+.1f} pts   "
          + ("CLEARS: the intervals are separated" if BEATS_COUNTING
             else "does NOT clear"))
    print(f"  vs one chain {(bon['accuracy'] - cot['accuracy']) * 100:+.1f} pts   "
          + ("clears" if BEATS_COT else "does NOT clear"))

    # The paired reading, on the items every configuration shares. It answers a narrower
    # question than the one above and answers it without the item sets differing.
    pm = max((interval(c, "majority", LADDERS_GRID) for c in LADDER_CONFIGS),
             key=lambda d: d["accuracy"])
    pb = max((interval(c, nm2, LADDERS_GRID) for c in LADDER_CONFIGS for nm2 in BON_NAMES),
             key=lambda d: d["accuracy"])
    GATE["paired"] = dict(n_items=len(GRID_ITEMS), n=GRID_N, majority=pm, best_of_n=pb,
                          delta_pts=(pb["accuracy"] - pm["accuracy"]) * 100,
                          separated=bool(pb["ci"][0] > pm["ci"][1]))
    print(f"\n  paired on the same {len(GRID_ITEMS)} items at N={GRID_N}: "
          f"counting {pm['accuracy']:.1%} ({pm['config_id']}) against best-of-N "
          f"{pb['accuracy']:.1%} ({pb['config_id']}), "
          f"{GATE['paired']['delta_pts']:+.1f} pts")

    if not BEATS_COUNTING:
        wide = bon["ci"][1] - bon["ci"][0] > 0.20
        print("\n  The next moves, in order, all measured rather than assumed:")
        if wide:
            print("   0. RESOLUTION FIRST. Best-of-N's own interval is wider than 20 points,")
            print("      so this is not yet a negative result. The configuration it won on")
            print(f"      carries {len(ITEMS_AT[cfg])} items; GENERATE fills it to the full")
            print("      set, and that is the cheapest thing on this list to fix.")
        print("   1. the next verifier tier in the knobs cell (a trained reward model)")
        print("   2. more N: the bar falls as coverage climbs and the vote saturates")
        print("   3. a hotter configuration, which lowers the bar (figure 2, right panel)")
        print("  If none of them clears it, the honest page says selection did not beat")
        print("  counting on this instrument, and shows the trap slice where it does.")
else:
    BEATS_COUNTING_CLEAN = False
    print("\nno verifier is scored, so the gate is not evaluated. Run RUN_MODE = 'SCORE'.")

GATE.update(beats_counting=BEATS_COUNTING, beats_one_chain=BEATS_COT)


## 13. Agreement with page 7

The vote at page 7's own configuration must reproduce page 7's number, because it is the same rule
on the same candidates. A disagreement means one of the two implementations has drifted, and it
would show up as two numbers for one procedure across a page break.

When page 8 has generated its own pool the check is weaker but still worth making: two independent
samples from one distribution should agree inside their intervals.


In [ ]:
PAGE_7_AGREEMENT = None
P7 = json.loads(P7_DERIVED.read_text()) if P7_DERIVED.exists() else None
if P7 is not None and "handoff" in P7:
    p7 = P7["handoff"]
    cid = P7.get("pool", {}).get("config_id")
    if cid in LADDERS:
        ours = LADDERS[cid]["ladder"]["majority"][-1]["accuracy"]
        cov_ours = LADDERS[cid]["ladder"]["coverage"][-1]["accuracy"]
        same_pool = POOL_SOURCE == "page_07"
        PAGE_7_AGREEMENT = dict(
            config_id=cid, n=LADDERS[cid]["axis"][-1], same_candidates=same_pool,
            page_07_vote=p7["plurality_accuracy"], page_08_vote=ours,
            page_07_coverage=p7["coverage"], page_08_coverage=cov_ours,
            vote_delta_pts=(ours - p7["plurality_accuracy"]) * 100)
        print(f"  {'':<14}{'page 7':>10}{'page 8':>10}{'delta':>9}")
        print(f"  {'the vote':<14}{p7['plurality_accuracy']:>10.1%}{ours:>10.1%}"
              f"{PAGE_7_AGREEMENT['vote_delta_pts']:>+9.1f}")
        print(f"  {'coverage':<14}{p7['coverage']:>10.1%}{cov_ours:>10.1%}"
              f"{(cov_ours - p7['coverage']) * 100:>+9.1f}")
        if same_pool and abs(PAGE_7_AGREEMENT["vote_delta_pts"]) > 0.5:
            print("\n!! Same rule, same candidates, two answers. One of the two vote")
            print("   implementations has drifted; fix it before either page ships.")
    else:
        print(f"page 7 names {cid} as its pool and this notebook has no ladder there")
else:
    print("out/page_07/derived.json not found or has no handoff; the cross-check is skipped")


## 14. The worked case

One problem where a rule picked a wrong candidate over a correct one that was sitting beside it.
Shown rather than asserted, and it is the single most persuasive artifact this page can carry: the
reader can read both chains and see what the rule rewarded.


In [ ]:
WORKED = {}
for rule in ["heuristic"] + BON_NAMES:
    fn = dict((n, f) for n, _, f in SELECTORS)[rule]
    cases = []
    for pid in ITEMS_AT[FIG3_CONFIG]:
        c = POOLS[FIG3_CONFIG][pid]
        i = fn(c, random.Random(0))
        if i is None or c[i]["correct"]:
            continue
        right = [x for x in c if x["correct"]]
        if not right:
            continue
        # The most legible specimen: the correct candidate the rule ranked furthest down.
        best_right = max(right, key=lambda x: len(x["trace"]))
        cases.append((len(best_right["trace"]) - len(c[i]["trace"]), pid, i,
                      best_right["sample_index"]))
    if not cases:
        print(f"{rule}: no problem where it picked a wrong candidate over a correct one")
        continue
    cases.sort(reverse=True)
    _, pid, i, ri = cases[0]
    c = POOLS[FIG3_CONFIG][pid]
    picked = c[i]
    passed = next(x for x in c if x["sample_index"] == ri)
    WORKED[rule] = dict(
        problem_id=pid, question=picked["question"], ground_truth=picked["ground_truth"],
        n_cases=len(cases),
        picked=dict(sample_index=picked["sample_index"], answer=picked["extracted_answer"],
                    correct=False, chars=len(picked["trace"]), trace=picked["trace"]),
        passed_over=dict(sample_index=ri, answer=passed["extracted_answer"], correct=True,
                         chars=len(passed["trace"]), trace=passed["trace"]))
    print(f"\n{'=' * 96}\n{rule}: {len(cases)} problems show this shape; the widest is below")
    print(textwrap.fill("Q: " + picked["question"], 92))
    print("ground truth:", picked["ground_truth"])
    for label, x in (("it picked", picked), ("it passed over", passed)):
        s = SCORES[(FIG3_CONFIG, pid, x["sample_index"])]
        extra = ""
        for f in (k for k in s if k.startswith("v:")):
            extra += f"   {f[2:]} {s[f]:+.2f}"
        print(f"\n  {label}: chain {x['sample_index']}   {len(x['trace'])} chars   "
              f"heuristic {s['heuristic']:.3f}{extra}   answer {x['extracted_answer']!r}   "
              f"{'CORRECT' if x['correct'] else 'wrong'}")
        print(textwrap.fill(x["trace"].replace("\n", " "), 92, initial_indent="    ",
                            subsequent_indent="    ")[:1200])


# Part 4 — Generation, and the artifacts

`GENERATE` is phase 1 and runs only after the gate in section 12 has been cleared. It is placed
last on purpose: everything above it works on a pool that already exists, so a reader can run this
notebook end to end without a GPU and still see what the experiment is.


## 15. Phase 1 — page 8's own pool

Its own `run_id`, the deck's instrument, and a temperature grid rather than a point.

**Independent requests, not one request for N answers.** Candidates listed inside a single response
condition on each other, which is precisely the diversity collapse page 7 is about, and it would
invalidate every curve here. `num_return_sequences` from one prompt is independent sampling and is
what this uses.

The sampler is stated here rather than imported, because what a candidate *is* is the claim this
page rests on, and a reader looking for it should find it on the page that depends on it.


In [ ]:
def config_id(temperature, cap):
    return f"t{temperature:.2f}-c{cap}"


def group_seed(problem_id, cid, group_index):
    """A stable seed per (problem, configuration, group), recorded on every record in the
    group, so a group regenerates exactly."""
    return int(harness.sha256_of([str(SEED), problem_id, cid, str(group_index)])[:15], 16)


@torch.inference_mode()
def sample_group(question, cap, temperature, k, seed, mdl, tok, eos_ids):
    """k independent chains for one problem, in one generate call."""
    assert temperature > 0, (
        "greedy decoding from one prompt has one answer, so k copies of it would be one "
        "chain pretending to be k. This page's pools are all sampled.")
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids
    inp = torch.tensor([prompt_ids], device=mdl.device)
    torch.manual_seed(seed)
    out = mdl.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=cap,
        # State every decoding parameter: a checkpoint can ship sampling defaults in its
        # generation_config, and then the manifest would record a setup never configured.
        do_sample=True, temperature=temperature, top_p=GEN["top_p"], top_k=None,
        num_return_sequences=k,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)

    chains = []
    for row in out:
        gen = row[inp.shape[1]:].tolist()
        # The first end-of-turn token is where this chain ended; everything after it is
        # padding to the longest row in the batch and is not content it wrote.
        stop = next((i for i, t in enumerate(gen) if t in eos_ids), None)
        truncated = stop is None
        gen = gen[:stop] if stop is not None else gen[:cap]
        text = tok.decode(gen, skip_special_tokens=True)
        chains.append(dict(trace=text, own_tokens=len(gen), truncated=truncated))
    return chains


if RUN_MODE == "GENERATE":
    from reasoning_from_scratch.ch03 import load_math500_test

    dev = get_device()
    got = harness.load_model(GEN["model_id"], device=dev, dtype=harness.pick_dtype(dev),
                             warmup_tokens=8)
    gmodel, gtok = got["model"], got["tokenizer"]
    harness.selftest(gtok)
    EOS_IDS = harness.end_of_turn_ids(gtok)

    pool_items = [p for p in load_math500_test()
                  if int(p["level"]) in (4, 5)][:GEN["n_items"]]
    ITEM_SET_HASH = harness.sha256_of([p["unique_id"] for p in pool_items])
    OWN_RUN_ID = (f"p08-{GEN['dataset']}-n{len(pool_items)}-{ITEM_SET_HASH[:8]}"
                  f"-cap{GEN['chain_cap']}-p{GEN['top_p']}-{dev.type}")
    gstore = harness.TraceStore(OWN_TRACES, OWN_RUN_ID,
                               ("model_id", "config_id", "problem_id", "sample_index"))
    gstore.report()

    made = 0
    for temperature, pool_n in GEN["grid"]:
        cid = config_id(temperature, GEN["chain_cap"])
        bar = harness.progress(pool_items, desc=f"T={temperature} x{pool_n}", unit="item")
        for row in bar:
            pid = row["unique_id"]
            have = [gstore.get(GEN["model_id"], cid, pid, s) for s in range(pool_n)]
            if all(h is not None for h in have):
                continue
            for start in range(0, pool_n, GEN["group"]):
                size = min(GEN["group"], pool_n - start)
                if all(gstore.get(GEN["model_id"], cid, pid, start + j) is not None
                       for j in range(size)):
                    continue
                g_index = start // GEN["group"]
                seed = group_seed(pid, cid, g_index)
                for j, g in enumerate(sample_group(row["problem"], GEN["chain_cap"],
                                                   temperature, size, seed, gmodel, gtok,
                                                   EOS_IDS)):
                    extracted = extract_final_candidate(g["trace"])
                    gstore.append({
                        "model_id": GEN["model_id"], "config_id": cid, "problem_id": pid,
                        "sample_index": start + j, "group_index": g_index,
                        "batch_seed": seed, "batch_size": size,
                        "level": int(row["level"]), "subject": row["subject"],
                        "question": row["problem"], "ground_truth": row["answer"],
                        "temperature": temperature, "top_p": GEN["top_p"],
                        "chain_budget": GEN["chain_cap"], "trace": g["trace"],
                        "own_tokens": g["own_tokens"], "tokens_generated": g["own_tokens"],
                        "truncated": g["truncated"],
                        "extracted_answer": extracted,
                        "vote_key": vote_key(g["trace"]),
                        "correct": bool(grade_answer(extracted, row["answer"])),
                    })
                    made += 1
            bar.set_postfix(gen=made)
    del gmodel, got
    harness.free_memory()
    print(f"\n{made:,} chains generated into {OWN_TRACES}")
    print("Now set POOL_SOURCE = 'own' and RUN_MODE = 'SCORE', and run from the top.")
else:
    print(f"RUN_MODE = {RUN_MODE!r}: nothing is generated. This page's own pool is made "
          f"only in GENERATE, and only after the section 12 gate.")


## 16. Artifacts

`derived.json` is the contract with the page: the aggregates it plots, the claims it may make, and
the manifest that says what produced them. The candidates are a download rather than a bundle.


In [ ]:
manifest = {
    "score_id": SCORE_ID,
    "notebook": "page_08_best_of_n.ipynb",
    "notebook_sha256": harness.file_sha256("page_08_best_of_n.ipynb"),
    "harness": harness.fingerprint(),
    "page": 8, "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "candidates": {"source": POOL_ORIGIN, "run_id": SOURCE_RUN_ID, "model_id": MODEL_ID,
                   "configs": [CONFIG_META[c] for c in LADDER_CONFIGS],
                   "chain_cap": CAP,
                   "n_items_at": {c: len(ITEMS_AT[c]) for c in LADDER_CONFIGS},
                   "grid_items": len(GRID_ITEMS), "main_config": MAIN_CONFIG,
                   "n_candidates": sum(len(POOLS[c][p]) for c in LADDER_CONFIGS
                                       for p in ITEMS_AT[c])},
    "verifiers": [dict(name=v["name"], kind=v["kind"],
                       model=MODEL_ID if v["name"] == "self" else v["model"],
                       quant=v.get("quant"),
                       solves_these_items=v.get("solves"),
                       stronger_generator_than_the_writer=(
                           v.get("solves") is not None and WRITER_SOLVES is not None
                           and v["solves"] > WRITER_SOLVES),
                       complete=any(h["name"] == v["name"] for h in HAVE),
                       scored_this_run=SCORED_THIS_RUN.get(v["name"], 0),
                       prompt=JUDGE_PROMPT if v["kind"] == "yesno" else None)
                  for v in ACTIVE],
    "subsampling": {"n_resamples": N_RESAMPLES, "seed": SEED,
                    "rule": "subsets without replacement, identical across selectors",
                    "score_tie_break": "lowest sample index",
                    "intervals": f"cluster bootstrap over items, {N_BOOTSTRAP} resamples"},
    "buckets": {"config_id": BUCKET_CONFIG, "holdout": BUCKET_HOLDOUT,
                "measuring_chains": BUCKET_N, "base_rate_if_circular": FULL_PILE_RATE},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

derived = {
    "manifest": manifest,
    # Read before any number on this page.
    "claims": {
        # The page's whole thesis. False means it may not be claimed.
        "best_of_n_beats_counting": BEATS_COUNTING,
        # The same claim with every verifier that is a stronger generator than the writer
        # removed. This is the one the page's title may quote.
        "best_of_n_beats_counting_without_a_stronger_generator": BEATS_COUNTING_CLEAN,
        "best_of_n_beats_one_chain": BEATS_COT,
        "verifier_scored": bool(HAVE),
        "bar_falls_with_temperature": bool(
            len(LADDER_CONFIGS) > 1
            and BAR[LADDER_CONFIGS[-1]][-1]["bar"] < BAR[LADDER_CONFIGS[0]][-1]["bar"]),
        "trap_bucket_plottable": TRAP_OK,
        "majority_declines_on_traps": bool(MAJORITY_DECLINES_ON_TRAPS),
        "buckets_are_held_out": BUCKET_HOLDOUT,
        "heuristic_is_a_length_ranking": HEURISTIC_IS_LENGTH,
        # False means the logprob diagnostic is mostly recovering the greedy chain.
        "logprob_is_more_than_greedy": (GREEDY_AGREEMENT is None
                                        or GREEDY_AGREEMENT["answer_agreement"] <= 0.8),
        "agrees_with_page_7": (PAGE_7_AGREEMENT is None
                               or not PAGE_7_AGREEMENT.get("same_candidates")
                               or abs(PAGE_7_AGREEMENT["vote_delta_pts"]) <= 0.5),
        "worked_case_exists": bool(WORKED),
        # The ablation's own claim: a judge WORSE at the problems than the writer still
        # beats counting, which is "verification is easier than generation" measured.
        "a_weaker_judge_beats_counting": bool(
            CHEAPEST_SUFFICIENT and WRITER_SOLVES
            and CHEAPEST_SUFFICIENT["solves"] < WRITER_SOLVES),
    },
    "gate": GATE,
    "bar": {c: BAR[c] for c in LADDER_CONFIGS},
    "bar_paired": {c: BAR_GRID[c] for c in LADDER_CONFIGS},
    "figure_1": {"config_id": FIG1_CONFIG, "axis": LADDERS[FIG1_CONFIG]["axis"],
                 "lines": [{"selector": n, "what": w,
                            "points": LADDERS[FIG1_CONFIG]["ladder"][n]}
                           for n, w, _ in HEADLINE]},
    "figure_2": {"grid_n": GRID_N, "grid_items": len(GRID_ITEMS),
                 "paired": True,
                 "by_temperature": [dict(CONFIG_META[c],
                                         **{k: at_n(c, k) for k in
                                            ("pass_at_1", "majority", "coverage", "bar",
                                             "bar_lo", "bar_hi")},
                                         best_of_n=(best_of_n_at(c, ladders=LADDERS_GRID)[1]
                                                    ["accuracy"] if BON_NAMES else None))
                                    for c in LADDER_CONFIGS],
                 "buckets": {"config_id": BUCKET_CONFIG, "base_rate": BASE_RATE,
                             "axis": BUCKET_AXIS,
                             "slices": {b: dict(n_items=d["n_items"], share=d["share"],
                                                ladder=d["ladder"])
                                        for b, d in BUCKET_LADDER.items()}}},
    "figure_3": {"config_id": FIG3_CONFIG, "bar": bar_here, "reliability": RELIABILITY,
                 "judge_ladder": JUDGE_LADDER,
                 "writer_solves": WRITER_SOLVES,
                 "cheapest_sufficient_judge": CHEAPEST_SUFFICIENT,
                 "discrimination": DISCRIMINATION, "worked_case": WORKED},
    "ladders": {c: LADDERS[c] for c in LADDER_CONFIGS},
    "ladders_paired": {c: LADDERS_GRID[c] for c in LADDER_CONFIGS},
    "page_7_agreement": PAGE_7_AGREEMENT,
    "logprob_vs_greedy": GREEDY_AGREEMENT,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"derived.json   {harness.kb(DERIVED_PATH):>10,.1f} KB   <- bundled with the page")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>10,.1f} KB   <- bundled with the page")
if SCORES_PATH.exists():
    print(f"scores.jsonl   {harness.kb(SCORES_PATH):>10,.1f} KB   <- published as a download")

# The blocking claim is the CLEAN one: a win that needed a judge better at the problems
# than the model being judged does not settle what this page says it settles.
blocking = ["verifier_scored", "best_of_n_beats_counting_without_a_stronger_generator",
            "agrees_with_page_7"]
failed = [k for k in blocking if not derived["claims"][k]]
print(f"\n{'TRANSLATABLE' if not failed else 'NOT TRANSLATABLE: ' + ', '.join(failed)}")
for k, v in derived["claims"].items():
    print(f"   {'ok ' if v else 'NO '} {k}")
print("\nThe blocking three are the page's thesis and its consistency with page 7. The rest")
print("change what the page SAYS rather than whether it can be written.")


## 17. Notes for re-derivation

- **The bar is measured before any verifier is loaded.** `majority(N) / coverage(N)` is fixed in
  section 3, and every later claim is checked against it. A threshold chosen after seeing the
  result is not a threshold.
- **The N axis is subsampled from a fixed-depth pool.** Page 7's N axis changes chain length as it
  changes N, because its budget is matched; this one does not, or the ladder would be measuring
  page 7's variable.
- **Every rule sees the same subsets**, drawn once per (item, configuration, N, repeat). Comparing
  rules across different draws would put sampling noise into the spread the figure reads as a
  difference between rules.
- **Intervals are over items, not over draws.** The draws inside one item are subsets of one pile
  and are not independent; treating them as independent trials makes the interval several times too
  narrow.
- **Reliability is defined only on items where a correct candidate exists.** On hopeless items every
  rule is wrong by construction, and including them would compress the differences the figure exists
  to show.
- **`self` is the control, not a contender.** A model grading its own work shares every error mode
  that produced it, and if it scores near the bar that is worth a sentence rather than a curve.
- **A judge that is a stronger generator proves the wrong thing.** Every tier records its own
  measured accuracy on these items, and the gate reports the best rung that is *not* stronger than
  the writer separately from the best rung overall. The first is the page's claim; the second is a
  ceiling.
- **A same-family judge is a weaker test than it looks.** All three tiers share Qwen2.5
  pretraining, so a judge that lands close to the self-judge is ambiguous between "size does not
  help" and "this training does not help", and settling that needs another family. The trained
  reward model tier is also the answer to this, since it is trained for a different job rather than
  scaled up at the same one.
- **The heuristic's weights are ch05's defaults** and are in the manifest. Changing them changes
  what "grades on presentation" means, so change them in the knobs and say so, or not at all.
- Ties among scores break to the lowest sample index. The vote's ties are not broken, because page 7
  does not break them.
- Temperature sampling is not reproducible across machines, so **the transcripts are the
  provenance**: the analysis is deterministic from them, and section 13 recomputes page 7's headline
  from the same file page 7 wrote.

What the page may claim is [`docs/page_8_measured_payoff.md`](../docs/page_8_measured_payoff.md),
checked at translation ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) section 6).
